# CohereX on Google Colab

Transcribe audio with word-level timestamps and (optionally) speaker labels.

Set the runtime to a GPU first: **Runtime → Change runtime type → GPU**.

In [ ]:
!apt-get -qq install -y ffmpeg
# 2>/dev/null hides Colab's harmless pip dependency-resolver warnings.
!pip install -q coherex 2>/dev/null

Log in to Hugging Face (needs access to the gated [Cohere Transcribe](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) model).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Upload an audio file.

In [ ]:
from google.colab import files
audio_path = next(iter(files.upload()))

Transcribe. Change `--language` to your audio's language, or use `--diarize` for speaker labels.

In [ ]:
!coherex "{audio_path}" --language en --vad_method silero --max_line_width 42 --max_line_count 2 -o out/

For Arabic, English, or Arabic-English code-switched audio, the finetuned [`cohere-transcribe-arabic-07-2026`](https://huggingface.co/CohereLabs/cohere-transcribe-arabic-07-2026) model is more accurate.

In [ ]:
!coherex "{audio_path}" --model CohereLabs/cohere-transcribe-arabic-07-2026 --language ar --vad_method silero --max_line_width 42 --max_line_count 2 -o out-ar/

Add speaker labels with `--diarize`. This needs access to the gated [`speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1) model (accept its terms first).

In [ ]:
from huggingface_hub import get_token
!coherex "{audio_path}" --language en --diarize --hf_token {get_token()} --vad_method silero --max_line_width 42 --max_line_count 2 -o out-diarize/

Outputs (SRT, VTT, TXT, TSV, JSON) are written to the `out/` folder in the file browser on the left.

---
## Optional: vLLM backend

Higher throughput for many/long files. It is **slower for a single short clip** (the server has to start and load the model first), so prefer the default backend above for quick jobs.

Run the install cell, then **Runtime → Restart session** (vLLM brings its own CUDA build), and finally run the last cell.

In [ ]:
# Installs vLLM with a matching CUDA build.
# After this finishes: Runtime -> Restart session, then run the next cell.
!uv pip install -q "coherex"

!uv pip install -U vllm==0.19.0 --torch-backend=auto
!uv pip install vllm[audio]
!uv pip install librosa==0.11.0

!uv pip install -U transformers==5.14.1 huggingface-hub==1.26.0 numpy==2.2
!uv pip install nvidia-cuda-runtime-cu12

In [ ]:
# Login to Hugging Face
# (needs access to the gated [Cohere Transcribe](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) model).

from huggingface_hub import notebook_login
notebook_login()

### Run a vLLM server for Cohere Transcribe

Start the server once with `vllm serve`, then reuse it for as many `coherex --vllm_url ...` runs as you like. It stays up until you stop it or the session ends.

In [ ]:
!nohup vllm serve CohereLabs/cohere-transcribe-arabic-07-2026 \
--gpu_memory_utilization 0.65 \
--trust-remote-code > vllm.log 2>&1 &

In [ ]:
# Point CohereX at the vLLM server started above (VAD + alignment still run locally).
# --model must match the model the server is serving.
import time, urllib.request

audio_path = "voice-sample-1.mp3"  # name of the file you uploaded above
VLLM_URL = "http://localhost:8000"

def vllm_ready():
    try:
        return urllib.request.urlopen(f"{VLLM_URL}/health", timeout=5).status == 200
    except Exception:
        return False

# Wait briefly for the server to finish loading the model.
for _ in range(75):  # ~375s
    if vllm_ready():
        break
    time.sleep(5)

if not vllm_ready():
    print("vLLM is not ready yet. The model is still loading — wait a bit and re-run this cell.")
    print("If it keeps failing, check the log for errors:  !tail -n 50 vllm.log")
else:
    !coherex "{audio_path}" \
        --model CohereLabs/cohere-transcribe-arabic-07-2026 --language ar \
        --backend vllm --vllm_url {VLLM_URL} \
        --vad_method silero --max_line_width 42 --max_line_count 2 -o out-vllm/